In [ ]:
print("\n" + "="*60)
print("CYCLE-WISE (LIFE-STAGE) ABLATION")
print("="*60)

life_rows = []

for target in TARGETS:
    D = RANDOM["results"][target]
    y_true = D["y_test"]
    y_pred = RANDOM["predictions"][(target, PROPOSED)]
    
    if target == 'SOH':
        stages = {
            'Early (>85%)': y_true > 85,
            'Mid (70-85%)': (y_true <= 85) & (y_true >= 70),
            'Late (<70%)': y_true < 70
        }
    else:
        stages = {
            'Early RUL (>100)': y_true > 100,
            'Mid RUL (30-100)': (y_true <= 100) & (y_true >= 30),
            'Late RUL (<30)': y_true < 30
        }
    
    for stage_name, mask in stages.items():
        if mask.sum() == 0: continue
        m = get_metrics(y_true[mask], y_pred[mask])
        life_rows.append({
            'Target': target,
            'Life Stage': stage_name,
            'Samples': int(mask.sum()),
            'RMSE': m['RMSE'],
            'MAE': m['MAE'],
            'R2': m['R2']
        })
        print(f"  {target} | {stage_name:20s} | RMSE={m['RMSE']:.4f}  R2={m['R2']:.4f}")

df_life = pd.DataFrame(life_rows)
display(df_life)
df_life.to_csv(out('Table_Life_Stage_Ablation.csv'), index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, t in enumerate(['SOH', 'RUL']):
    sub = df_life[df_life['Target'] == t]
    bars = axes[i].bar(sub['Life Stage'], sub['RMSE'], color=['#2ca02c', '#ff7f0e', '#d62728'], edgecolor='black')
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                     f'{bar.get_height():.3f}', ha='center', fontweight='bold')
    axes[i].set_title(f'{t}: RMSE across Life Stages', fontweight='bold')
    axes[i].set_ylabel('RMSE')
    axes[i].grid(axis='y', ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig(out('Fig_Life_Stage_Ablation.png'), dpi=DPI)
plt.show()
print(f"✅ Saved: Table_Life_Stage_Ablation.csv and Fig_Life_Stage_Ablation.png")